# LC 312 — Burst Balloons

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Think in reverse — instead of
asking "which balloon to burst first?" ask "which balloon is
burst <em>last</em> in the interval (i, j)?" That last balloon's
neighbors are always `nums[i]` and `nums[j]`, making the
subproblem independent and enabling interval DP.
</div>

## Official Problem Statement

Given `n` balloons indexed `0` to `n-1`, each with number `nums[i]`.
Burst balloon `i` to gain `nums[i-1] * nums[i] * nums[i+1]` coins
(using 1 for out-of-bounds). Return the maximum coins you can
collect by bursting all balloons.

**Example 1:**
```
Input:  nums = [3,1,5,8]
Output: 167
Explanation:
  nums=[3,1,5,8] → burst 1: 3*1*5=15 → [3,5,8]
  → burst 5: 3*5*8=120 → [3,8]
  → burst 3: 1*3*8=24  → [8]
  → burst 8: 1*8*1=8
  Total: 15+120+24+8=167
```
**Constraints:**
- `n == nums.length`
- `1 <= n <= 300`
- `0 <= nums[i] <= 100`

## What This Is Actually Asking

Bursting balloons in order changes the neighbors of remaining
balloons, making the problem order-dependent and hard to reason
forward. The greedy approach (always burst the smallest) fails.

The reversal insight: pad `nums` with 1s on both ends to handle
boundaries. Define `dp[i][j]` as the max coins from bursting all
balloons strictly between indices `i` and `j`.

For each interval `(i, j)`, try each `k` as the LAST balloon
burst. Since `k` is last, its neighbors at burst time are
exactly `nums[i]` and `nums[j]` — subproblems `(i, k)` and
`(k, j)` are already solved and independent.

## Walk Through an Example by Hand

```
nums = [3,1,5,8]  → padded: [1,3,1,5,8,1]
                              0 1 2 3 4 5

dp[i][j] = max coins bursting all between i and j

Length 2 intervals (1 balloon each):
  dp[0][2]: k=1. coins=nums[0]*nums[1]*nums[2]=1*3*1=3
  dp[1][3]: k=2. coins=nums[1]*nums[2]*nums[3]=3*1*5=15
  dp[2][4]: k=3. coins=nums[2]*nums[3]*nums[4]=1*5*8=40
  dp[3][5]: k=4. coins=nums[3]*nums[4]*nums[5]=5*8*1=40

Length 3 intervals (2 balloons):
  dp[0][3]: k=1: dp[0][1]+1*3*5+dp[1][3]=0+15+15=30
            k=2: dp[0][2]+1*1*5+dp[2][3]=3+5+0=8
            → dp[0][3]=30
  dp[1][4]: k=2: dp[1][2]+3*1*8+dp[2][4]=0+24+40=64
            k=3: dp[1][3]+3*5*8+dp[3][4]=15+120+0=135
            → dp[1][4]=135
  dp[2][5]: k=3: dp[2][3]+1*5*1+dp[3][5]=0+5+40=45
            k=4: dp[2][4]+1*8*1+dp[4][5]=40+8+0=48
            → dp[2][5]=48

dp[0][5]: final answer
  k=1: 0+1*3*1+dp[1][5]=... try all k
  → dp[0][5]=167 ✓
```

## The Picture

```
Padded nums: [1, 3, 1, 5, 8, 1]
              0  1  2  3  4  5

dp[i][j] = best coins from open interval (i, j)

Filling order: by interval length (small → large)

  length=2: dp[0][2], dp[1][3], dp[2][4], dp[3][5]
  length=3: dp[0][3], dp[1][4], dp[2][5]
  length=4: dp[0][4], dp[1][5]
  length=5: dp[0][5]  ← ANSWER

For dp[i][j], try each k in (i+1, j-1) as LAST burst:

  dp[i][j] = max over k of:
    dp[i][k]              ← coins from left subproblem
  + nums[i]*nums[k]*nums[j]  ← coins for k as last burst
  + dp[k][j]              ← coins from right subproblem

Why it works: k is last, so its neighbors are fixed at i,j.
```

## When To Use This Pattern

- When a problem's subproblems interfere with each other in
  the forward direction, think **reverse the choice order**.
- When asking "which to remove/pick last?" decouples
  subproblems, think **interval DP with last-choice enumeration**.
- When you see "open intervals" with boundary values determining
  the cost, think **pad boundaries with neutral values** (1s here).
- When a problem has n<=300 and O(n³) is acceptable, think
  **interval DP** (Matrix Chain, MCM, Stone Merge).
- When subproblems are "all elements strictly between i and j",
  think **dp[i][j] over open intervals**.

## The Approach

Pad `nums` with 1 on both ends. Create a 2D `dp` table where
`dp[i][j]` = max coins from bursting all balloons strictly
between `i` and `j`.

Fill the table by increasing interval length (length 2 first,
then 3, up to n+1). For each interval `(i, j)`, try each `k`
strictly inside as the last balloon burst, and take the max of
`dp[i][k] + nums[i]*nums[k]*nums[j] + dp[k][j]`.

The answer is `dp[0][n+1]` where `n` is the original length.

In [ ]:
# Imports
from typing import List

In [ ]:
# ----------------------------------------------------------
# Harness
# ----------------------------------------------------------
def test_harness(func):
    cases = [
        ([3,1,5,8],   167),
        ([1,5],        10),
        ([1],           1),
        ([0,0,0],       0),
        ([7,9,8,0,7],  1498),
    ]
    passed = 0
    for nums, expected in cases:
        result = func(nums)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if not ok:
            print(f"{status} | nums={nums}"
                  f" | expected={expected} | got={result}")
        else:
            print(f"{status} | nums={nums}"
                  f" | coins={result}")
        passed += ok
    print(f"\n{passed}/{len(cases)} tests passed")

In [ ]:
def maxCoins(nums: List[int]) -> int:
    """
    LC 312 — Burst Balloons

    Interval DP:
    - Pad nums with 1 on both ends
    - dp[i][j] = max coins from open interval (i,j)
    - For each interval, try each k as last burst:
        dp[i][j] = max(dp[i][k]
                       + nums[i]*nums[k]*nums[j]
                       + dp[k][j])
    - Fill by increasing length
    - Answer: dp[0][n+1]

    Time:  O(n^3)
    Space: O(n^2)
    """
    pass
    # Debug hints:
    # print(f"i={i} j={j} k={k}"
    #       f" val={dp[i][k]+nums[i]*nums[k]*nums[j]+dp[k][j]}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxCoins)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (all permutations) | O(n!) | O(n) | Completely infeasible |
| Backtracking + memo | O(n^3) | O(n^2) | Same as DP |
| Interval DP (optimal) | O(n^3) | O(n^2) | n<=300 → ~27M ops OK |
| Greedy (burst min first) | — | — | WRONG approach |

## Real World Connection

**Finance / DE context:** The "last choice decouples subproblems"
insight appears in financial option pricing: the decision of
when to exercise last determines the boundary conditions for
earlier decisions, enabling backward induction (the standard
technique in lattice pricing models at Citi).

Interval DP also underlies query optimization in databases —
the order in which you join tables determines cost, and
dynamic programming over join order intervals is exactly how
PostgreSQL and AWS Athena choose join plans.

In data compression and Huffman-like tree construction, the
"optimal merge order" problems reduce to interval DP with
boundary conditions — the same core pattern.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra